In [ ]:
import os

from style import *

os.chdir(ROOT)  # run from the project root, so .env and data/ are found
apply_style()
banner(
    "Shelfmate",
    "Recommends your next 5 books, links you to Indian stores, and runs on your own machine.",
)

from shelfmate.logging_setup import new_run, setup_logging

setup_logging("notebook")
new_run("notebook")

In [ ]:
section(1, "The data", "About 6 million ratings from 53,424 readers for 10,000 books")
import pandas as pd
from sqlalchemy import create_engine

from shelfmate.config import settings

books = pd.read_csv("data/raw/books.csv").rename(columns={"authors": "author"})
titles = books.set_index("book_id")["title"]

engine = create_engine(settings.database_url)
ratings = pd.read_sql("SELECT user_id, book_id, rating FROM ratings", engine)
# The loader was run twice in this project, so drop the duplicate pairs first.
ratings = ratings.drop_duplicates(["user_id", "book_id"])

n_users, n_books = ratings["user_id"].nunique(), ratings["book_id"].nunique()
stat_tiles(
    {
        "Ratings": f"{len(ratings):,}",
        "Readers": f"{n_users:,}",
        "Books": f"{n_books:,}",
        "Empty cells": f"{1 - len(ratings) / (n_users * n_books):.1%}",
    }
)

In [ ]:
RED_RISING_ID = int(
    books.loc[books["title"].str.contains("Red Rising", regex=False), "book_id"].iloc[0]
)
show(
    books.loc[books["book_id"] == RED_RISING_ID, ["book_id", "title", "author"]],
    "Starting from this book",
)

In [ ]:
section(
    2,
    "People who liked it also liked",
    "Collaborative filtering with ALS (Phase 2, Part A)",
)
from shelfmate.recommender.collaborative import build_matrix, recommend_for_me, train

user_items = build_matrix(ratings)
model = train(user_items)
cf_top = recommend_for_me(
    model, my_liked_ids=[RED_RISING_ID], n_books=user_items.shape[1], n=50
)
show(
    pd.DataFrame(
        {"title": [titles[b] for b, _ in cf_top], "score": [s for _, s in cf_top]}
    ),
    "Top 5 from other readers' taste",
    n=5,
)

In [ ]:
section(
    3,
    "Books that read alike",
    "Each title becomes 384 numbers, and close numbers mean similar books (Phase 2, Part B)",
)
import numpy as np

from shelfmate.recommender.content import embed

# The guide embeds Open Library descriptions. Titles and authors keep this cell quick.
texts = (books["title"] + ". " + books["author"]).tolist()
emb = embed(texts)  # 10,000 x 384
red = emb[books.index[books["book_id"] == RED_RISING_ID][0]]
content_scores = emb @ red  # cosine similarity, because the vectors are normalised

closest = np.argsort(content_scores)[::-1][1:6]
show(
    pd.DataFrame(
        {
            "title": books["title"].iloc[closest].values,
            "similarity": content_scores[closest],
        }
    ),
    "Closest by text",
)

In [ ]:
section(
    4,
    "The blend",
    "With few ratings the content score counts for 70% and the collaborative score for 30% (Phase 2, Part C)",
)
from shelfmate.recommender.hybrid import hybrid

cf = {int(b): float(s) for b, s in cf_top}
order = np.argsort(content_scores)[::-1][:50]
content = {int(books["book_id"].iloc[i]): float(content_scores[i]) for i in order}

ranked = hybrid(cf, content, n_rated=1, n=50)
ranked = [(b, s) for b, s in ranked if b != RED_RISING_ID]

top = ranked[:10][::-1]
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh([titles[b][:42] for b, _ in top], [s for _, s in top], height=0.55, color=BLUE)
label_bars(ax, "{:.2f}", only_max=True, horizontal=True)
finish(
    ax,
    "Hybrid ranking",
    "Higher means a stronger match for a Red Rising reader",
    x_grid=True,
)
ax.grid(False, axis="y")
plt.show()

In [ ]:
section(
    5,
    "Two lanes",
    "Continue the series you started, and try something new (Phase 2, Part E)",
)
from shelfmate.recommender.lanes import next_in_series, parse_series, something_new

parsed = books["title"].map(parse_series)
books["series_name"] = [name for name, _ in parsed]
books["series_pos"] = [pos for _, pos in parsed]
books_by_id = books.set_index("book_id")

read = books_by_id.loc[[RED_RISING_ID]].assign(
    rating=5
)  # pretend you read Red Rising and liked it
first_book = (
    books.dropna(subset=["series_name"])
    .sort_values("series_pos")
    .groupby("series_name")["book_id"]
    .first()
    .to_dict()
)

keep_going = [titles[b] for b in next_in_series(read, books_by_id)]
new = [
    titles[b] for b, _ in something_new(ranked, books_by_id, {"Red Rising"}, first_book)
]
show(
    pd.DataFrame(
        {
            "lane": ["Continue your series"] * len(keep_going)
            + ["Something new"] * len(new),
            "title": keep_going + new,
        }
    ),
    "Your next books",
)

In [ ]:
section(
    6,
    "Does it beat a simple baseline?",
    "Hide 20% of 2,000 readers' liked books and see how many we find (Phase 2, Part D)",
)
from scipy.sparse import csr_matrix

liked = ratings[ratings["rating"] >= 4]
per_user = liked.groupby("user_id")["book_id"].apply(list)
test_users = per_user[per_user.apply(len) >= 20].sample(2000, random_state=42)

rng = np.random.default_rng(42)
hidden = {}
for uid, liked_books in test_users.items():
    liked_books = list(liked_books)
    rng.shuffle(liked_books)
    hidden[uid] = set(liked_books[: max(1, int(0.2 * len(liked_books)))])

drop = [(u, b) for u, hs in hidden.items() for b in hs]
train_liked = liked[~liked.set_index(["user_id", "book_id"]).index.isin(drop)]
train_matrix = csr_matrix(
    (
        np.ones(len(train_liked), dtype=np.float32),
        (train_liked["user_id"], train_liked["book_id"]),
    ),
    shape=user_items.shape,
)
eval_model = train(train_matrix)

als, base = [], []
popular = np.asarray(train_matrix.sum(axis=0)).ravel().argsort()[::-1]
for uid, hs in hidden.items():
    ids, _ = eval_model.recommend(
        uid, train_matrix[uid], N=5, filter_already_liked_items=True
    )
    als.append(len(set(ids) & hs) / 5)
    seen = set(train_matrix[uid].indices)
    rec = [b for b in popular if b not in seen][:5]
    base.append(len(set(rec) & hs) / 5)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(
    ["Popularity", "ALS"],
    [np.mean(base), np.mean(als)],
    width=0.45,
    color=[ORANGE, BLUE],
)
label_bars(ax, "{:.3f}")
finish(
    ax,
    "The model beats the baseline",
    "precision@5 on 2,000 test readers. Higher is better.",
)
plt.show()

In [ ]:
section(
    7, "Store links", "Search links for one book, built without any web calls (Phase 3)"
)
from shelfmate.agent.search import NEW_STORES, store_links

show(
    pd.DataFrame(store_links("Golden Son", "Pierce Brown", NEW_STORES)),
    "Store links for Golden Son",
)

In [ ]:
section(
    8,
    "Ask the agent",
    "A local language model picks the tools and writes the answer (Phase 3, needs Ollama running)",
)
from shelfmate.agent.graph import ask

result = ask("I finished Golden Son, 5 stars. What next?")
for m in result["messages"]:
    m.pretty_print()

In [ ]:
section(9, "Email the result", "Sends a real email, so it is commented out (Phase 4)")
# from shelfmate.notify import send_email
# send_email("Shelfmate notebook run", "<p>The notebook finished.</p>")

In [ ]:
banner(
    "That is the whole system",
    "Data in PostgreSQL, a hybrid recommender, two lanes, a price search and an agent, all in one place.",
)